<center><h1>HW3</h1></center>
<br>
<br>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [17]:
import os,re,glob
import pandas as pd
import numpy as np
from sklearn.feature_selection import f_classif

Get the AReM Data Set

In [2]:
DATA_DIR = "../data/AReM"  
COLS = ["time", "avg_rss12", "var_rss12", "avg_rss13",
        "var_rss13", "avg_rss23", "var_rss23"]

def read_one(path):
    """Read one AReM file: skip '#' comment lines, handle comma/space
    delimiters and trailing commas, and drop the time column."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p for p in re.split(r"[,\s]+", line) if p]
            rows.append([float(p) for p in parts[:7]])
    return pd.DataFrame(rows, columns=COLS).drop(columns="time")

def file_num(path):
    """Extract the integer file index, e.g. 'dataset10.csv' -> 10."""
    return int(re.findall(r"\d+", os.path.basename(path))[0])

records = []
for act in sorted(os.listdir(DATA_DIR)):
    folder = os.path.join(DATA_DIR, act)
    if not os.path.isdir(folder):
        continue 
    for fp in sorted(glob.glob(os.path.join(folder, "dataset*.csv")), key=file_num):
        records.append({
            "activity": act,
            "file_id": file_num(fp),
            "data": read_one(fp),  # 480 x 6 DataFrame
        })

print("Total instances:", len(records))
lengths = pd.Series([len(r["data"]) for r in records])
print("Rows per file:\n", lengths.value_counts())
print("Any missing values:", any(r["data"].isna().any().any() for r in records))


Total instances: 88
Rows per file:
 480    87
479     1
Name: count, dtype: int64
Any missing values: False


In [3]:
short = [(r["activity"], r["file_id"], len(r["data"])) for r in records if len(r["data"]) != 480]
print("Files with fewer than 480 rows:", short)

Files with fewer than 480 rows: [('sitting', 8, 479)]


### (b) Test and Train Data

In [4]:
def is_test(activity, file_id):
    """Return True if this instance belongs to the test set (rule from 1(b))."""
    test_ids = {1, 2} if activity.startswith("bending") else {1, 2, 3}
    return file_id in test_ids

for r in records:
    r["split"] = "test" if is_test(r["activity"], r["file_id"]) else "train"

train_records = [r for r in records if r["split"] == "train"]
test_records  = [r for r in records if r["split"] == "test"]

print("Train instances:", len(train_records))
print("Test instances: ", len(test_records), "\n")
print(pd.crosstab(pd.Series([r["activity"] for r in records], name="activity"),
                  pd.Series([r["split"] for r in records], name="split")))

Train instances: 69
Test instances:  19 

split     test  train
activity             
bending1     2      5
bending2     2      4
cycling      3     12
lying        3     12
sitting      3     12
standing     3     12
walking      3     12


### (c) Feature Extraction

#### i. Research

#####  **Commonly used time-domain features**

**1. Central tendency**
- Mean, median, mode

**2. Dispersion / variability**
- Standard deviation, variance
- Range (max − min), interquartile range (IQR = Q3 − Q1)
- Mean / median absolute deviation (MAD)
- Coefficient of variation (std / mean)

**3. Order statistics**
- Minimum, maximum
- Quartiles (Q1, Q3) and other percentiles (e.g., 10th, 90th)

**4. Distribution shape**
- Skewness (asymmetry of the distribution)
- Kurtosis (heaviness of the tails)

**5. Energy and amplitude**
- Energy (sum of squared values), root mean square (RMS)
- Mean absolute value, peak-to-peak amplitude

**6. Temporal structure / signal change**
- Mean of first differences, waveform length (sum of absolute differences)
- Zero-crossing rate or mean-crossing rate
- Number of peaks
- Autocorrelation at different lags
- Slope of a linear trend fitted over time

**7. Complexity**
- Entropy measures (e.g., sample entropy, approximate entropy)
- Hjorth parameters (activity, mobility, complexity)


#### ii. Extraction

In [5]:
STAT_NAMES = ["min", "max", "mean", "median", "std", "1st_quart", "3rd_quart"]
META_COLS = ["activity", "file_id", "split"]

def series_features(x):
    x = np.asarray(x, dtype=float)
    return {
        "min": x.min(),
        "max": x.max(),
        "mean": x.mean(),
        "median": np.median(x),
        "std": x.std(ddof = 1),
        "1st_quart": np.percentile(x,25),
        "3rd_quart": np.percentile(x,75),
    }
    

def extract_features(records, n_segments = 1):
    rows = []
    for r in records:
        row = {
            "activity": r["activity"],
            "file_id": r["file_id"],
            "split": r["split"]
        }
        values = r["data"].to_numpy()
        segments = np.array_split(values, n_segments, axis = 0)
        for s,seg in enumerate(segments):
            for j in range(seg.shape[1]):
                ts_id = s*seg.shape[1] + j + 1
                for name,val in series_features(seg[:,j]).items():
                    row[f"{name}{ts_id}"] = val
        rows.append(row)

    df = pd.DataFrame(rows)
    df.index = pd.RangeIndex(1, len(df) + 1, name="Instance")
    return df

features = extract_features(records)
feature_cols = [c for c in features.columns if c not in META_COLS]

print("Feature table shape:", features.shape)
print("Number of features:", len(feature_cols))

pd.set_option("display.max_columns", 50)
features.round(3)


Feature table shape: (88, 45)
Number of features: 42


,activity,file_id,split,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,bending1,1,test,37.25,45.00,40.625,40.50,1.477,39.25,42.000,0.0,1.30,0.359,0.430,0.323,0.00,0.500,4.00,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.000,1.120,27.25,38.25,34.311,35.00,2.188,33.000,36.00,0.00,1.92,0.571,0.43,0.583,0.000,1.300
2,bending1,2,test,38.00,45.67,42.813,42.50,1.436,42.00,43.670,0.0,1.22,0.372,0.470,0.289,0.00,0.500,2.00,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.000,1.145,27.67,38.50,33.025,33.00,1.995,32.000,34.50,0.00,3.11,0.571,0.43,0.601,0.000,1.300
3,bending1,3,train,35.00,47.40,43.954,44.33,1.559,43.00,45.000,0.0,1.70,0.426,0.470,0.339,0.00,0.500,6.50,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.000,0.830,29.00,38.50,35.588,36.00,2.000,35.362,36.50,0.00,1.79,0.493,0.43,0.514,0.000,0.940
4,bending1,4,train,33.00,47.75,42.180,43.50,3.671,39.15,45.000,0.0,3.00,0.696,0.500,0.631,0.00,1.120,8.50,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.430,1.300,20.00,38.67,33.494,35.00,3.849,30.458,36.33,0.00,2.18,0.614,0.50,0.524,0.000,1.000
5,bending1,5,train,33.00,45.75,41.678,41.75,2.243,41.33,42.750,0.0,2.83,0.536,0.500,0.405,0.43,0.710,3.00,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.430,1.120,23.67,37.50,29.857,30.00,2.411,28.458,31.25,0.00,1.79,0.383,0.43,0.389,0.000,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,walking,11,train,19.50,45.33,33.587,34.25,4.651,30.25,37.000,0.0,14.67,4.577,4.260,2.461,2.87,6.105,7.33,23.25,15.318,15.330,2.970,13.25,17.270,0.0,9.00,3.227,3.10,1.647,2.050,4.260,6.33,25.00,15.892,16.00,3.284,13.730,18.25,0.00,8.32,3.260,3.11,1.640,2.050,4.322
85,walking,12,train,19.75,45.50,34.323,35.25,4.752,31.00,38.000,0.0,13.47,4.456,3.900,2.599,2.49,5.970,6.25,22.25,15.210,15.250,2.711,13.50,17.000,0.0,9.00,3.277,3.11,1.500,2.170,4.390,6.25,23.25,15.550,15.75,3.120,13.500,17.75,0.00,9.67,3.433,3.20,1.733,2.157,4.565
86,walking,13,train,19.50,46.00,34.546,35.25,4.842,31.25,37.812,0.0,12.47,4.372,4.135,2.360,2.49,5.800,7.00,22.67,15.186,15.250,2.758,13.25,17.000,0.0,8.34,3.029,2.86,1.665,1.790,4.110,7.00,22.75,15.760,15.75,2.823,14.000,17.75,0.00,10.00,3.338,3.08,1.657,2.160,4.335


#### iii. Standard Deviation

In [13]:
feature_std = features[feature_cols].std(ddof = 1)
feature_std.round(4)

min1          9.5700
max1          4.3944
mean1         5.3357
median1       5.4401
std1          1.7722
1st_quart1    6.1536
3rd_quart1    5.1389
min2          0.0000
max2          5.0627
mean2         1.5742
median2       1.4122
std2          0.8841
1st_quart2    0.9464
3rd_quart2    2.1253
min3          2.9565
max3          4.8751
mean3         4.0084
median3       4.0364
std3          0.9467
1st_quart3    4.2207
3rd_quart3    4.1716
min4          0.0000
max4          2.1836
mean4         1.1661
median4       1.1456
std4          0.4582
1st_quart4    0.8436
3rd_quart4    1.5525
min5          6.1240
max5          5.7412
mean5         5.6756
median5       5.8138
std5          1.0249
1st_quart5    6.0965
3rd_quart5    5.5317
min6          0.0458
max6          2.5189
mean6         1.1548
median6       1.0865
std6          0.5176
1st_quart6    0.7586
3rd_quart6    1.5236
dtype: float64

In [15]:
def bootstrap_std_ci(x, n_boot = 1000, alpha = 0.10, seed = 56):
    rng = np.random.default_rng(seed)
    x =  np.asarray(x, dtype = float)
    n = len(x)
    
    boot_stds = np.empty(n_boot)
    for b in range(n_boot):
        sample = rng.choice(x, size = n, replace = True)
        boot_stds[b] = sample.std(ddof = 1)
        
    lower = np.percentile(boot_stds, 100*alpha/2)
    upper = np.percentile(boot_stds, 100*(1-alpha/2))
    return x.std(ddof = 1), lower, upper 


ci_rows = []
for col in feature_cols:
    s, lo, up = bootstrap_std_ci(features[col])
    ci_rows.append({"feature": col, "std": s, "CI_lower": lo, "CI_upper":up})

ci_table = pd.DataFrame(ci_rows).set_index("feature")
ci_table.round(4)

,std,CI_lower,CI_upper
feature,,,
min1,9.5700,8.1855,10.8147
max1,4.3944,3.3055,5.3035
mean1,5.3357,4.6563,5.8797
median1,5.4401,4.7462,6.0191
std1,1.7722,1.5631,1.9515
1st_quart1,6.1536,5.5452,6.6421
3rd_quart1,5.1389,4.2475,5.8292
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6318,5.3853


#### iv. Select Features

In [20]:
train_feat = features[features["split"] == "train"]
X_train = train_feat[feature_cols]
y_train = train_feat["activity"]

F, p = f_classif(X_train, y_train)

f_scores = pd.Series(F, index = feature_cols ).fillna(0)
f_scores.sort_values(ascending = False).round(2)

f_df = pd.DataFrame({
    "stat":   [c[:-1] for c in f_scores.index],       
    "series": [int(c[-1]) for c in f_scores.index],   
    "F":      f_scores.values,
})

f_table = f_df.pivot(index="stat", columns="series", values="F").loc[STAT_NAMES]
f_table.round(2)

f_rank = f_table.rank(axis=0, ascending=False)   # rank the 7 statistics within each series (1 = best)
f_rank["mean_rank"] = f_rank.mean(axis=1)        # average rank of each statistic over the 6 series
f_rank.sort_values("mean_rank")

/Users/non-sense/miniconda3/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [ 7 21] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/non-sense/miniconda3/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


series,1,2,3,4,5,6,mean_rank
stat,,,,,,,
mean,3.0,1.0,3.0,2.0,2.0,2.0,2.166667
3rd_quart,6.0,2.0,2.0,3.0,3.0,1.0,2.833333
median,2.0,3.0,6.0,1.0,4.0,3.0,3.166667
1st_quart,1.0,5.0,5.0,4.0,5.0,4.0,4.000000
max,7.0,6.0,1.0,6.0,1.0,6.0,4.500000
std,5.0,4.0,7.0,5.0,7.0,5.0,5.500000
min,4.0,7.0,4.0,7.0,6.0,7.0,5.833333


In [21]:
corrs = []
for k in range(1, 7):
    cols = [f"{s}{k}" for s in STAT_NAMES]
    c = X_train[cols].corr()
    c.index = STAT_NAMES
    c.columns = STAT_NAMES
    corrs.append(c)

avg_corr = pd.concat(corrs).groupby(level=0).mean().loc[STAT_NAMES, STAT_NAMES]
avg_corr.round(2)

,min,max,mean,median,std,1st_quart,3rd_quart
min,1.00,0.36,0.59,0.57,-0.32,0.60,0.52
max,0.36,1.00,0.79,0.75,0.63,0.71,0.82
mean,0.59,0.79,1.00,0.99,0.38,0.98,0.98
median,0.57,0.75,0.99,1.00,0.36,0.98,0.96
std,-0.32,0.63,0.38,0.36,1.00,0.28,0.50
1st_quart,0.60,0.71,0.98,0.98,0.28,1.00,0.93
3rd_quart,0.52,0.82,0.98,0.96,0.50,0.93,1.00


**Selected features: 
mean, standard deviation, and maximum.**

To judge which features are most useful for distinguishing activities, we
computed the one-way ANOVA F-statistic of each of the 42 features on the
**training set only** (to avoid using test data in a modeling decision).
A larger F means a larger difference between activity means relative to the
spread within each activity. Because F-values from the variance series
(2, 4, 6) are much larger than those from the average series (1, 3, 5),
we ranked the 7 statistics within each series and averaged the ranks.
We also computed the average correlation between the statistics across the
6 series to check for redundancy.

Findings:
- **mean** has the best average rank (2.17) and is in the top 3 for every
  series, making it the most consistent feature.
- mean, median, 1st quartile, and 3rd quartile are almost perfectly
  correlated (r ≥ 0.98). They carry nearly the same information, so only one
  of them (mean) is kept.
- **std** has the lowest correlation with mean (r = 0.38), so it adds
  complementary information about variability. Its F-values on the variance
  series are high (about 129–361), higher than those of any max feature,
  although its within-series rank appears low because the location
  statistics on these series are even stronger.
- **max** ranks first on series 3 and 5 (avg_rss13, avg_rss23), where mean and
  std are weaker, and is only moderately correlated with mean (0.79) and std
  (0.63).
- **min** is the weakest statistic (average rank 5.83). On the variance series
  it is almost always 0, and `min2` and `min4` are constant across all
  instances, so it is not selected.

Therefore, mean, std, and max are chosen because together they describe three
different aspects of each time series: its location, its variability, and its
extreme values.

## 2. ISLR 3.7.4

### (a) Linear Train

### (b) Linear Test

### (c) Not Linear Train

### (d) Not Linear Testing

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 